In [2]:
import pandas as pd, numpy as np, os, glob, datetime 
from nltk.tokenize import sent_tokenize

# Load original unitised manifesto data
* Version 2024-1 of Manifesto Project Corpus 

Lehmann, Pola / Franzmann, Simon / Al-Gaddooa, Denise / Burst, Tobias / Ivanusch, Christoph / Regel, Sven / Riethmüller, Felicia / Volkens, Andrea / Weßels, Bernhard / Zehnter, Lisa (2024): The Manifesto Data Collection. Manifesto Project (MRG/CMP/MARPOR). Version 2024a. Berlin: Wissenschaftszentrum Berlin für Sozialforschung (WZB) / Göttingen: Institut für Demokratieforschung (IfDem). https://doi.org/10.25522/manifesto.mpds.2024a

In [3]:
import os
os.chdir("C:/Users/Sanford/Documents/eiee/capable/manifestos/rep_package/data/2024_corpus")

In [4]:
master = pd.DataFrame()
codes = []
for x in glob.glob('normal_unitised/*.csv'):
    print(x)
    country = x.split('\\')[1].split('_')[0]
    print(country)
    df = pd.read_csv(x)
    print(df.shape[0])
    df = df.drop(columns=['Unnamed: 0'])
    df['country']=country
    master = pd.concat([master,df])


normal_unitised\Argentina_unitised_manifestos.csv
Argentina
16251
normal_unitised\Australia_unitised_manifestos.csv
Australia
73734
normal_unitised\Austria_unitised_manifestos.csv
Austria
45352
normal_unitised\Belgium_unitised_manifestos.csv
Belgium
180758
normal_unitised\Bolivia_unitised_manifestos.csv
Bolivia
7895
normal_unitised\Brazil_unitised_manifestos.csv
Brazil
41517
normal_unitised\Bulgaria_unitised_manifestos.csv
Bulgaria
12362
normal_unitised\Canada_unitised_manifestos.csv
Canada
34351
normal_unitised\Chile_unitised_manifestos.csv
Chile
54562
normal_unitised\Colombia_unitised_manifestos.csv
Colombia
27493
normal_unitised\Costa Rica_unitised_manifestos.csv
Costa Rica
47740
normal_unitised\Croatia_unitised_manifestos.csv
Croatia
31161
normal_unitised\Cyprus_unitised_manifestos.csv
Cyprus
12655
normal_unitised\Czech Republic_unitised_manifestos.csv
Czech Republic
26224
normal_unitised\Denmark_unitised_manifestos.csv
Denmark
18910
normal_unitised\Dominican Republic_unitised_mani

C:\Users\Sanford\AppData\Local\Temp\ipykernel_35936\2137256816.py:7: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(x)


149540
normal_unitised\Sweden_unitised_manifestos.csv
Sweden
22004
normal_unitised\Switzerland_unitised_manifestos.csv
Switzerland
23387
normal_unitised\United Kingdom_unitised_manifestos.csv
United Kingdom
35029
normal_unitised\United States_unitised_manifestos.csv
United States
16708
normal_unitised\Uruguay_unitised_manifestos.csv
Uruguay
16606


In [5]:
master.shape

(1808645, 11)

In [6]:
master.head()

,text,cmp_code,eu_code,pos,manifesto_id,party,date,language,annotations,translation_en,country
0,PROMETIMOS CAMBIAR LA HISTORIA Y LA HEMOS CAMB...,H,NaN,1,150201_199505,150201,199505,spanish,True,True,Argentina
1,La legislación electoral exige para la oficial...,000,NaN,2,150201_199505,150201,199505,spanish,True,True,Argentina
2,"En verdad, para nosotros los justicialistas, n...",305.1,NaN,3,150201_199505,150201,199505,spanish,True,True,Argentina
3,Nuestro dilema era cambiar la historia o perde...,601.1,NaN,4,150201_199505,150201,199505,spanish,True,True,Argentina
4,"Al asumir el gobierno, nos encontramos con una...",606.1,NaN,5,150201_199505,150201,199505,spanish,True,True,Argentina


In [7]:
master.columns

Index(['text', 'cmp_code', 'eu_code', 'pos', 'manifesto_id', 'party', 'date',
       'language', 'annotations', 'translation_en', 'country'],
      dtype='object')

In [ ]:
# master[master['party']==92435].manifesto_id.unique()

array(['92435_200710', '92435_201110', '92435_201510'], dtype=object)

In [8]:
print('Unique, quasi-sentence unitised manifestos:',len(master.manifesto_id.unique()))

Unique, quasi-sentence unitised manifestos: 1571


In [9]:
master['post_hoc_unit']=0

In [10]:
master['sentence_num']=None

In [11]:
master.country.value_counts()

country
Belgium               180758
Spain                 149540
Netherlands           118512
Germany                88694
Norway                 85170
Australia              73734
Chile                  54562
Mexico                 50341
New Zealand            49202
Portugal               48136
Costa Rica             47740
Hungary                47666
Austria                45352
Lithuania              44980
Ireland                44915
Brazil                 41517
Slovenia               40029
Israel                 39294
United Kingdom         35029
Greece                 34763
Canada                 34351
Croatia                31161
Luxembourg             30845
Slovakia               30480
Colombia               27493
Poland                 27461
Czech Republic         26224
Switzerland            23387
Italy                  22623
Finland                22562
Sweden                 22004
Panama                 21062
Denmark                18910
Estonia                16813
United

In [12]:
master.language.value_counts()

language
spanish       390457
english       239302
dutch         201762
german        178937
french        132085
portuguese     89653
norwegian      85170
hungarian      47666
greek          47418
lithuanian     44980
slovenian      40029
hebrew         39270
croatian       31161
slovak         30257
polish         26930
czech          26224
catalan        23180
swedish        22286
finnish        22280
italian        22177
danish         18910
estonian       16813
bulgarian      12362
romanian       11339
galician        5966
latvian         2031
Name: count, dtype: int64

In [13]:
# # create QS id 

# Group by manifesto_id and create counter within each group
master2 = master.reset_index(drop=True)

# Then use the vectorized approach
master2['counter'] = master2.groupby('manifesto_id').cumcount()
master2['qs_new'] = master2['manifesto_id'] + '_' + master2['counter'].astype(str).str.zfill(4)
master2 = master2.drop('counter', axis=1)

In [ ]:
# Need to clean the cmp codes -- some mixed types in the original codes, e.g., string '500', string '500.0', and float 500.0 
# First copy original codes for safekeeping
master2['cmp_code_orig'] = master2['cmp_code']

# convert all empty or missing cmp codes to 0
master2.loc[master2['cmp_code']=='','cmp_code']=0
master2.loc[master2['cmp_code'].isna(),'cmp_code']=0

# Then need to preserve the strings ('H') and standardise the numbers
clean_codes = []
for i in master2['cmp_code']:
    try:
        new_i = i.astype(float)
        clean_codes.append(new_i)   
    except:
        #print(i)
        clean_codes.append(i)


In [ ]:
pd.Series(clean_codes).value_counts()[:20]

504.0    161397
411.0    126941
503.0    103563
H         92644
501.0     90121
506.0     89280
0.0       85454
303.0     71095
403.0     57727
701.0     54909
502.0     49487
301.0     42491
402.0     38660
605.1     37255
416.2     36676
107.0     34884
410.0     31130
202.1     28915
605.0     27730
703.1     26859
Name: count, dtype: int64

In [17]:
master2['cmp_code_clean']=clean_codes

In [18]:
master2.columns

Index(['text', 'cmp_code', 'eu_code', 'pos', 'manifesto_id', 'party', 'date',
       'language', 'annotations', 'translation_en', 'country', 'post_hoc_unit',
       'sentence_num', 'qs_new', 'cmp_code_orig', 'cmp_code_clean'],
      dtype='object')

In [19]:
master2.to_csv('combined_normal_unitised.csv',header=True,index=False)

# Load post-hoc unitised manifestos (returned as text dumps from the Manifesto Project API)

In [20]:
add = pd.DataFrame()
codes = []
for x in glob.glob('post_hoc_unitised/*.csv'):
    print(x)
    country = x.split('\\')[1].split('_')[0]
    print(country)
    df = pd.read_csv(x)
    print(df.shape[0])
    df = df.drop(columns=['Unnamed: 0'])
    df['country']=country
    add = pd.concat([add,df])


post_hoc_unitised\Australia_63320_199003_post_hoc_unitised_manifestos.csv
Australia
448
post_hoc_unitised\Australia_63320_199303_post_hoc_unitised_manifestos.csv
Australia
578
post_hoc_unitised\Australia_63320_199603_post_hoc_unitised_manifestos.csv
Australia
2976
post_hoc_unitised\Australia_63320_199810_post_hoc_unitised_manifestos.csv
Australia
161
post_hoc_unitised\Australia_63320_200111_post_hoc_unitised_manifestos.csv
Australia
222
post_hoc_unitised\Australia_63321_199603_post_hoc_unitised_manifestos.csv
Australia
456
post_hoc_unitised\Australia_63321_199810_post_hoc_unitised_manifestos.csv
Australia
97
post_hoc_unitised\Australia_63321_200111_post_hoc_unitised_manifestos.csv
Australia
265
post_hoc_unitised\Australia_63620_199003_post_hoc_unitised_manifestos.csv
Australia
534
post_hoc_unitised\Australia_63620_199303_post_hoc_unitised_manifestos.csv
Australia
397
post_hoc_unitised\Australia_63620_199810_post_hoc_unitised_manifestos.csv
Australia
81
post_hoc_unitised\Australia_63810

In [21]:
add.shape

(272012, 12)

In [22]:
add['post_hoc_unit']=1

In [23]:
add.columns

Index(['text', 'cmp_code', 'eu_code', 'pos', 'manifesto_id', 'party', 'date',
       'language', 'annotations', 'translation_en', 'sentence_num', 'country',
       'post_hoc_unit'],
      dtype='object')

In [24]:
add[add['party']==92435].manifesto_id.unique()

array(['92435_200509'], dtype=object)

In [25]:
# # create QS id 

# Group by manifesto_id and create counter within each group
add2 = add.reset_index(drop=True)

# Then use the vectorized approach
add2['counter'] = add2.groupby('manifesto_id').cumcount()
add2['qs_new'] = add2['manifesto_id'] + '_' + add2['counter'].astype(str).str.zfill(4)
add2 = add2.drop('counter', axis=1)

In [26]:
add2.to_csv('combined_post_hoc_unitised.csv',header=True,index=False)

In [27]:
add2.shape

(272012, 14)

In [28]:
add2.manifesto_id.unique().shape

(370,)

## To immediately accommodate the un-unitised texts into the full sample, need to pass them through the manifestoberta model
* script 1a_manifestoberta.ipynb
* yields set of post-hoc unitised manifestos with cmp copdes: combined_post_hoc_unitised_CMP_codes.csv 

In [ ]:
# load in sentences with codes

add_codes = pd.read_csv('combined_post_hoc_unitised_CMP_codes.csv')

## Concat the two together

In [ ]:
all = pd.concat([master2,add_codes])

In [30]:
all.reset_index(inplace=True)

In [31]:
all['year'] = all['date'].astype(str).str[:4].astype(int)

In [35]:
all.columns

Index(['index', 'text', 'cmp_code', 'eu_code', 'pos', 'manifesto_id', 'party',
       'date', 'language', 'annotations', 'translation_en', 'country',
       'post_hoc_unit', 'sentence_num', 'qs_new', 'cmp_code_orig',
       'cmp_code_clean', 'year'],
      dtype='object')

## fix slashes
* double slashes demark quasi-sentences in the annotated pdfs
* some have not been parsed correctly so we fix that manually here

In [ ]:
# First, identify QS with slashes. Could be 
sl_all = pd.Series([1 if '//' in x.strip('//') else 0 for x in all['text']])
sl_all.sum()

In [ ]:
# Export for inspection 
all['slash']=sl_all
pd.DataFrame([all[all['slash']==1]['qs_new'],all[all['slash']==1]['original_text']]).T.to_csv('bads.csv')

# Manually inspect and confirm which units actually contain multiple QS and therefore need to be separated
# Not all units containing slashes need to be separated -- some are URLs or are truly one QS but the slashes
# are at the end, demarking the boundary
# add 'fix_slash' column denoting these to be fixed

In [ ]:
# Load in fixed quasi-sentences
tofix = pd.read_csv('bads_w_fix2.csv')
fslsh = tofix[tofix['fix_slash']==1]
fslsh.shape

In [ ]:
# initiate new df to house split entries
new_entries = pd.DataFrame()
# iterate over qs that need to be fixed
for qs,text in zip(fslsh['qs_new'],fslsh['original_text']):
    # split text
    splits = text.split('//')
    #print(splits)
    
    # enumerate over splits
    for n,split in enumerate(splits):
        # check for empties 
        if len(split) < 1: 
            continue
            
        # extrct original from database, create new entries based on the original meta data but with new text and new qs 
        else:
            new = all[all['qs_new']==qs]
            new['original_text']=split
            new['qs_new']=new['qs_new']+f".{n}"

            # compile into new entries dataframe
            new_entries = pd.concat([new_entries,new])

orig_ind = new_entries.index
new_entries.reset_index(inplace=True,drop=True)

In [ ]:
new_entries.shape

In [ ]:
orig_ind = orig_ind.unique()
orig_ind

In [ ]:
# concat new entries with master
all2=pd.concat([all,new_entries])
all2.shape

In [ ]:
# drop original (bad) qs
all2.drop(orig_ind,inplace=True)
all2.shape

In [ ]:
# reset index of all2
all2.reset_index(inplace=True,drop=True)

In [ ]:
#### remove slashes from rest of the ok ones

In [ ]:
# remove slashes
slsh = tofix.query('remove_slash == 1')
for i in slsh['qs_new']:
    t = d[d['qs_new']==i]['original_text'].tolist()[0]
    t = t.replace('//','')
    d.loc[d['qs_new']==i,'original_text']=t

## Clean special chars

In [32]:
# Function to replace HTML entities and specific codes without regex issues
def clean_special_characters(text):
    # Handle specific HTML codes (e.g. &nbsp;, &amp;, etc.)
    html_entity_patterns = {
        '&nbsp;': ' ',  # Replace non-breaking space with a normal space
        '&amp;': '&',    # Replace &amp; with &
        '&#39;': "'",    # Replace HTML apostrophe entity with actual apostrophe
        '\xa0': ' ',     # Replace non-breaking space \xa0 with a normal space
        # Add more specific replacements as needed
    }
    
    # Replace each HTML entity with its corresponding replacement
    for entity, replacement in html_entity_patterns.items():
        text = text.replace(entity, replacement)

    remove = ['\uf0b7','•','\uf034','«','»', '//|','@', '#','*','\xa0','\uf02f']#'\uf02f',\uf020 

    for rem in remove:
        text = text.replace(rem, '')    
    
    return text.strip()


In [ ]:
# Apply the cleaning function to the DataFrame
all2['text_cleaned'] = all2['text'].apply(clean_special_characters)


## Count words

In [40]:
# Function to count the words in a string
def count_words(text):
    # Split the text by whitespace to get words, and return the length of the list
    words = text.split()
    return len(words)

# Apply the word counting function to the DataFrame column
all2['word_count'] = all2['text_cleaned'].apply(count_words)


## Audit and separate the QS that are too long 
* Through an iterative manual process, we inspected all QS exceeding a word length of 200 words.
* This revealed that some were table of contents, urls, introductions and text boxes that the Manifesto Project didn't include in their annotations, and bad parses. We cross-reference these with the manifesto pdfs and classify them accordingly. Some which contain no meaningful text (the urls or bad parses, sometimes filled with numbers from tables or other graphics), are marked to delete. 

In [ ]:
n = 199
all2.query(f'word_count == {n}')['text_cleaned'].tolist()

In [41]:
intro = ['162410_201410_0000', '51210_201505_0004', '51210_201505_0003', '171210_201506_0003', '11520_202209_0002','62623_201510_0001', '64620_199911_0002','12320_200109_0002','51901_201505_0011','12951_200909_0003',
       	'51621_201505_0005', 	'43320_201510_0005',	'51902_201505_0003','64110_201409_3015','63320_201607_4270','63320_201607_5044','63110_201905_0001','62320_201510_0003','62320_201105_0001','171305_201506_0002',
         '180410_198911_0002','51340_201505_0002','11420_202209_0002','180410_198911_0001']  

lists = ['63620_199810_0000', '63810_199810_0000', '35220_199510_0566', '35220_199510_0712', '35220_199510_0598', '35220_199510_0778', '35220_199510_0525', '64621_199911_0313', '32110_199403_0264', '35220_199510_1141',
                '35220_199510_0770', '35220_199510_0320', '35220_199510_0898', '32110_199403_0192', '35220_199510_1026', '35220_199510_1077', '35220_199510_0355', '35220_199510_0695', '35220_199510_0706',
                '35220_199510_1170', '35220_199510_0956', '35220_199510_1044', '35220_199510_1130', '35229_200502_0520', '64621_199911_0078', '35220_199510_1115', '35220_199510_0581', '64621_199911_0435', '35220_199510_0333',
                '35220_199510_0403', '64621_199911_0396', '35220_199510_0705', '35220_199510_1107', '35220_199510_0822', '35220_199510_0942', '35220_199510_0481', '35220_199510_0640', '35220_199510_0999', '35220_199510_0935',
                '35220_199510_0240', '35220_199510_0877', '35220_199510_1194', '63620_199810_0083', '63810_199810_0083', '35229_200502_1289', '35220_199510_0983', '32110_199403_0168', '35220_199510_0681', '63620_199810_0075',
                '63810_199810_0075', '12810_200109_2772', '35220_199510_0436', '35220_199510_0335', '63620_199810_0024', '63810_199810_0024', '35220_199510_0833', '35220_199510_0539', '35220_199510_0393', '35220_199510_0672', 
                '35220_199510_1090', '35229_200502_0349', '35220_199510_0943', '35220_199110_0213', '35220_199510_0844', '35220_199510_0305', '35220_199110_0015', '35220_199510_0287', '35220_199510_0303', '35220_199510_1070', 
                '35220_199510_1206', '35220_199510_1061', '35220_199510_0387', '35220_199510_0272', '35220_199510_0871', '35220_199510_0542', '35220_199510_0591', '32110_199403_0220', '63810_199810_0007', '63620_199810_0007',
                '152622_200605_1958', '152622_200605_7728', '35220_199510_1064', '35220_199510_0896', '35220_199510_0998', '35229_200502_0518', '35220_199510_0690', '35220_199510_0241', '35220_199510_0257', '35220_199510_0367',
                '35220_199510_0469', '35220_199510_0546', '35220_199510_0651', '35220_199510_0676', '35220_199510_0709', '35220_199510_0816', '35220_199510_0912', '35220_199510_0933', '35220_199510_1203', 
                '64621_199911_0248', '64621_199911_0322', '35229_200502_1233', '35220_199510_0954', '64621_199911_0520', '160221_201405_1523', '35220_199110_0187', '62620_199310_0341', '35229_200502_1070', 
                '64621_199911_0271', '35220_199110_0013', '35229_200502_0048', '63320_199303_0172', '35220_199510_0256', '152622_200605_6668','63320_199303_0108','35229_200502_1291','35220_199510_0582','64621_199911_0077',
                '35220_199510_0955','35229_200502_0170']  

bad_parse = ['113441_202004_2107','180309_198911_0186']

box_info =  ['21111_201905_1583', '51901_201912_0046', '21426_201905_0344', '63320_201607_6159', '21111_201905_3099', '150029_198905_0530', '62420_200601_0916', '63320_201607_4093', '63320_201607_4867',
                 '21111_201905_2714', '21912_201905_3157', '64110_201409_0806', '63320_201607_4425', '63320_201607_5199', '63320_201607_3949', '63320_201607_4723', '63320_201607_5847', '63320_201607_5613', '21111_201905_1220',
                 '63320_201607_5401', '63320_201905_2965', '12520_201309_3149', '12520_200909_1862', '160221_201905_0966', '64110_201409_1379', '21111_201905_2569','21111_201905_1357','63320_201905_3456',
                 '21111_201905_3175','63320_201607_2526']

dell = ['21912_201905_0001', '64320_200207_3096', '64320_200207_3095', '62901_200011_2198', '180620_202210_0003', '12620_200909_0013', '12221_200109_0023', '62420_199706_0017', '62901_200011_2199', '62901_200011_2195', '12951_200109_0000',
 '35229_200502_0001', '62420_199706_1946', '22526_200205_0003', '171311_201506_0002', '180311_202210_0414', '21112_199505_0004', '12320_200909_0006', '12810_200909_0004', '51621_201505_0003', '13520_200111_0550', '64320_200207_3097', 
 '12810_200909_0002', '33902_199603_0005', '12810_200909_0005', '180410_198911_0000', '32111_200105_1135', '32213_200105_1135', '74712_201506_0368', '33902_199603_0002', '32110_199403_0008','171301_201506_0001','43110_201510_0002','35229_200502_0001']



## deal with the problematic entries

In [42]:
for id in dell:
    all2.loc[all2['qs_new']==id,'del']=1

### deal with intros
* new units in new_intro_units_zip

In [43]:
# FIRST, the intros are easy --> just update cmp code to 'intro'
for i in intro:
    all2.loc[all2['qs_new']==i,'cmp_code']='intro'

In [ ]:
# but lets actually tokenize these just in case
new_intro_units = []
for i in intro:
    t = all2[all2['qs_new']==i].text_cleaned.iloc[0]
    t_new = sent_tokenize(t)
    new_intro_units.append(t_new)
    #print(len(t_new),t_new)
    #break

In [ ]:
new_intro_units_zip = zip(intro,new_intro_units)

### deal with lists
* new units in new_lists_units_zip

In [ ]:
# NEXT, try to handle lists and box_info. try detecting ';','•','*','-'
pat = r'[;•*]'
new_lists_units = []
for i in lists:
    sub = re.split(pat, all2.query(f'qs_new == "{i}"').text.iloc[0])
    #print(i)
    keep = []
    for chunk in sub:
        if len(chunk.strip())>0: keep.append(chunk.strip())
    new_lists_units.append(keep)


# this appears to work, so need to add these into the dataset and remove the originals (how it is done for the //, see code above)

In [ ]:
new_lists_units_zip = zip(lists,new_lists_units)

### deal with text boxes/tables 
* new units in: new_box_units_zip 

In [ ]:
# NEXT, the text boxes/tables
all2[all2['qs_new'].isin(box_info)]#.text.tolist()


# all of these have been ignored by the manifesto coders so safe to say we can? for now put them as 'ignored'
all2[all2['qs_new'].isin(box_info)].text.tolist()

['• 4 CHIFFRES À RETENIR Selon le Réseau Transition, en mars 2018, le nombre d’initiatives de transition en Wallonie et à Bruxelles est passé de 35 à 120 en deux ans. Il y en a deux ou trois nouvelles qui démarrent chaque mois. Toutes ont vocation a être soutenues. Certaines développent des projets économiques inspirants, des entreprises qui créent de nouveaux moyens de subsistance, et montrent de nouvelles façons de fournir des biens et services essentiels qui répondent à un vrai besoin local.• Même si le taux de création d’entreprise reste positif, 10.742 faillites ont été enregistrées en Belgique en 2017. 4.956 en Région flamande, 2.778 en Région wallonne, 2.698 en Région bruxelloise et 310 d’entreprises étrangères, soit une hausse de 6,72% par rapport à 2016.• Selon une étude du Syndicat Neutre des Indépendants de 2016, 16,6% des indépendants gagnent moins de 833 € net par mois et un indépendant sur 6 vit sous le seuil de pauvreté.• Selon une étude de l’ULB sur le bien-être des com

In [ ]:
# update their cmp code
for i in box_info:
    all2.loc[all2['qs_new']==i,'cmp_code']='box'

In [ ]:
# but lets actually tokenize these just in case
new_box_units = []
for i in box_info:
    t = all2[all2['qs_new']==i].text_cleaned.iloc[0]
    t_new = sent_tokenize(t)
    new_box_units.append(t_new)

In [ ]:
new_box_units_zip = zip(box_info,new_box_units)

### now deal with the bad parses
* as it turns out, all of the worst ones of these are from a single manifesto (the austrian greens in 1990). However, this party received just 4.78% of the vote and therefore technically fall outside our criteria. 
* will remove this manifesto from the dataset 
* still have two to deal with 

In [ ]:
# NEXT, the base parses. these are tricky because some just lack puncutation
all2[all2['qs_new'].isin(bad_parse)].text.tolist()

['지역별 핵심공약 서울특별시「서울형 대기질 개선」 추진하여 미세먼지 걱정 없는 서울을 만들겠습니다. 버스·택시 등 전기차 보급 및 충전소 인프라 확충. 전기차(2019년 1.3만대→2022년 10만대), 수소차(2019년 500대→2022년 3,000대) 등 친환경차 보급 확대 및 건설기계 매연저감장치 부착 등 저공해화미세먼지 계절관리제 시행, 행정·공공기관 출입차량 2부제, 에코마일리지 시즌제 도입, 미세먼지 간이측정기 확대로 대기질 통합 정보 확대신혼부부 주거지원 확대 및 청년수당 지원 도입으로 청년이 행복한 서울을 만들겠습니다.서울시 신혼부부 주거지원 사업(전세대출 융자, 주택공급 확대 등) 3년 간 총 3조원 투입, 연간 2만5천쌍 지원만19~34세 서울 거주하는 졸업 후 2년 간 미취업 청년 3만명 대상, 1인당 50만원씩 6개월 간 수당 지급시설물 노후화 대비 빅데이터 시스템 개발 등 종합관리를 통해 시민이 안전한 서울을 만들겠습니다. 서울시 주요기반시설(교량, 터널, 상·하수도, 하천 등) 점검·보수·보강·성능개선 등 이력정보를 빅데이터화한 분석 시스템 개발 및 종합관리계획 구축제로페이를 정착·확산시켜 자영업자·소상공인이 행복한 서울을 만들겠습니다.제로페이 가맹점 추가 확보 및 소비자·판매자 편의 위한 결제시스템 개선온누리상품권·지역화폐 제로페이 포인트 충전 등 소비자 이용활성화 위한 민관협력 추진2032년 서울-평양하계올림픽 공동 개최로 남북이 하나되는 서울을 만들겠습니다. 동북아 긴장 완화, 한반도 평화를 위한 2032년 서울-평양 하계 올림픽 공동 개최 유치 추진부산광역시광역교통망 확충으로 부·울·경 1시간대 생활권을 구축하겠습니다. 부산~김해 간 대저대교, 엄궁대교 신속 착공 및 조기 개통 추진부전~마산 복선철도를 조속 개통하고 광역전철 운행과 부·울·경 광역환승할인제 도입 적극 추진국제관광도시 부산의 얼굴로 북항과 원도심 일대를 재구성하겠습니다.2030년 부산월드엑스포 유치 범정부 차원 적극 추진북항 통합개발사업을 원도심 대개조사업과 연

In [ ]:
# but lets actually tokenize these just in case
new_bp_units = []
for i in bad_parse:
    t = all2[all2['qs_new']==i].text_cleaned.iloc[0]
    t_new = sent_tokenize(t)
    new_bp_units.append(t_new)

In [ ]:
new_bp_units_zip = zip(bad_parse,new_bp_units)

## now need to add the newly unitised QS and remove the originals

In [ ]:
total = []
totalids=[]
zips = [new_box_units_zip,new_lists_units_zip,new_intro_units_zip,new_bp_units_zip]
for z in zips:
    #print(z)
    for p in z:
        #print(p)
        total.append(p)
        totalids.append(p[0])

In [ ]:
uns = set()
n=0
for t,l in total:
    uns.add(t)
    n+=len(l)
n, len(uns)

(2113, 175)

In [ ]:
len(uns)

175

In [ ]:
# initiate new df to house split entries
new_entries = pd.DataFrame()
# iterate over qs that need to be fixed
for qs,units in total:
    # split text

    # enumerate over splits
    for n,split in enumerate(units):
        # extrct original from database, create new entries based on the original meta data but with new text and new qs     
        new = all2[all2['qs_new']==qs]
        new['text_cleaned']=split # CANNOT BE JUST 'TEXT' BECAUSE THEN WITH THE LATER CLEANING IT DOES NOT GET PICKED UP -- TEXT_CLEANED
        new['qs_new']=new['qs_new']+f".{n}"

        # compile into new entries dataframe
        new_entries = pd.concat([new_entries,new])

#orig_ind = new_entries.index
new_entries.reset_index(inplace=True,drop=True)

C:\Users\Sanford\AppData\Local\Temp\ipykernel_44704\3471910737.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new['text_cleaned']=split # CANNOT BE JUST 'TEXT' BECAUSE THEN WITH THE LATER CLEANING IT DOES NOT GET PICKED UP -- TEXT_CLEANED
C:\Users\Sanford\AppData\Local\Temp\ipykernel_44704\3471910737.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new['qs_new']=new['qs_new']+f".{n}"


In [ ]:
#new_entries
# concat new entries with original
all3=pd.concat([all2,new_entries])
all3.shape

(756431, 190)

In [ ]:
# Remove the bad ones 
all3=all3[~all3['qs_new'].isin(totalids)]


In [ ]:
# create dummy for short QS, headings, boxes, intros, 0s 

keep = all3.apply(lambda r: 1 if r['word_count']>2 | (r['cmp_code_clean']!='H') | (r['cmp_code_orig']!='H') 
        | (r['cmp_code_clean']!='box') | (r['cmp_code_clean']!='intro') else 0, axis=1)

all3['keep']=keep

In [ ]:
all3.reset_index(drop=True,inplace=True)

## Pervote inspection
* For the initial collections, we used the per_vote variable to subset the parties with more than 5% vote share. However, not all manifestos for parties that gained 5% have a coded per_vote value due to how this variable is constructed (see the Manifesto Project Codebook). So in subsequent collections, we did not use this variable to download the data and manually verified that parties excluded from our analysis did not indeed have less than 5% vote share, adding manifestos for parties that were initially excluded as necessary. As a result, the dataset does include some parties with less 5% vote share but should not exclude parties that gained more than 5% even when this data was not provided by the Manifesto Project Dataset.  

## Keyword detection steps
* As described in the paper, we curated the training set over rounds of sampling the manifesto data based on keywords and random selection. We include below the code used to do the keyword identification. Note: This initial training set only included the manifestos of EU countries that had already been unitised. We manually validated post-hoc the performance of the model on the languages that we added aftewards. 
* We then exported samples per languages of quasi-sentences with keyword matches, excluding sentences coded as headings by the Manifesto Team and those with less than 3 words. We then took a sample of quasi-sentences without keyword matches per language. 
* We then proceeded with manual annotations across 4 coders of the training set, with each sentence seen by at least two coders. All disagreements were then aligned by discussion with the full team.
* Once the initial training set was finalised, we proceed with the model training. 
  

In [ ]:
# kw_utils contains some helper functions and all of the translations for our target keywords 
from kw_utils import get_kw, extract_keywords, cohen_k

In [ ]:
# load dictionaries and extraction function from kw_utils
kw_clim_em_ren = get_kw('clim_em_ren')
kw_sust_env = get_kw('sust_env')

In [ ]:
# identify all matches se and clim em ren matches
all_matches = pd.DataFrame()

for lang in all3['language'].unique():
    print(lang)
    s = all3[all3['language']==lang]
    #print(s.shape)

    kws = kw_sust_env[lang]
    matches = extract_keywords(s['text'],kws,list=True)
    s['kw_se']=matches

    kws = kw_clim_em_ren[lang]
    matches = extract_keywords(s['text'],kws,list=True)
    s['kw_clim_em_ren']=matches

    s=s[['qs_new','kw_se','kw_clim_em_ren','language']]
    all_matches=pd.concat([all_matches,s])

In [ ]:
all_matches.drop(columns=['language'],inplace=True)

In [ ]:
# merge back with master dataset
all4 = pd.merge(all3,all_matches,on='qs_new',how='outer')#['sampled'].sum()

In [ ]:
all4.to_csv('full_cleaned_manifesto_texts.csv',header=True,index=False)